In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot
import numpy as np
import urllib.request
import zipfile
import os

In [6]:
# ============================================================
# LOAD DATA
# ============================================================

In [7]:

# UCI SMS Spam Collection dataset
# Source: https://archive.ics.uci.edu/dataset/228/sms+spam+collection
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "smsspamcollection.zip"

# Download the zip file
urllib.request.urlretrieve(url, zip_path)

# Extract it
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("sms_spam_data")

# Check extracted files
print(os.listdir("sms_spam_data"))

# Load the tab-separated file (no header, 2 columns: label, message)
df = pd.read_csv("sms_spam_data/SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])

# Quick look
print(df.shape)
print(df.head())
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True))  # check class imbalance

['readme', 'SMSSpamCollection']
(5572, 2)
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64
label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64


In [8]:
# ============================================================
# EDA
# ============================================================

In [9]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

Missing values:
 label      0
message    0
dtype: int64


In [10]:

# Check for duplicate messages
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate messages only:", df.duplicated(subset='message').sum())



Duplicate rows: 403
Duplicate messages only: 403


In [11]:
# Message length analysis (character count) — spam messages are typically longer
df['message_length'] = df['message'].apply(len)
print("\nMessage length stats by label:")
print(df.groupby('label')['message_length'].describe())


Message length stats by label:
        count        mean        std   min    25%    50%    75%    max
label                                                                 
ham    4825.0   71.482487  58.440652   2.0   33.0   52.0   93.0  910.0
spam    747.0  138.670683  28.873603  13.0  133.0  149.0  157.0  223.0


In [14]:
# Word count
df['word_count'] = df['message'].apply(lambda x: len(x.split()))
print("\nWord count stats by label:")
print(df.groupby('label')['word_count'].describe())


Word count stats by label:
        count       mean        std  min   25%   50%   75%    max
label                                                            
ham    4825.0  14.310259  11.517945  1.0   7.0  11.0  19.0  171.0
spam    747.0  23.911647   5.780174  2.0  22.0  25.0  28.0   35.0


In [15]:
df.head()

,label,message,message_length,word_count
0,ham,"Go until jurong point, crazy.. Available only ...",111,20
1,ham,Ok lar... Joking wif u oni...,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,28
3,ham,U dun say so early hor... U c already then say...,49,11
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,13


In [16]:
# ============================================================
# EDA - DUPLICATE ANALYSIS
# ============================================================

In [17]:
# Check label distribution within duplicates before dropping
print(df[df.duplicated(subset='message', keep=False)]['label'].value_counts(normalize=True))

label
ham     0.73538
spam    0.26462
Name: proportion, dtype: float64


In [18]:
# Drop duplicate messages (data leakage risk otherwise)
df = df.drop_duplicates(subset='message', keep='first').reset_index(drop=True)

df.shape
df['label'].value_counts(normalize=True)

label
ham     0.87367
spam    0.12633
Name: proportion, dtype: float64

In [19]:
df.head()

,label,message,message_length,word_count
0,ham,"Go until jurong point, crazy.. Available only ...",111,20
1,ham,Ok lar... Joking wif u oni...,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,28
3,ham,U dun say so early hor... U c already then say...,49,11
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,13


In [20]:
# ============================================================
# TRAIN/TEST SPLIT
# ============================================================

In [21]:
from sklearn.model_selection import train_test_split

# Split BEFORE text preprocessing/vectorization to prevent data leakage
# (vectorizer must be fit only on train data, same principle as StandardScaler)
X = df['message']
y = df['label'].map({'ham': 0, 'spam': 1})  # encode target: ham=0, spam=1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("\nTrain label distribution:\n", y_train.value_counts(normalize=True))
print("\nTest label distribution:\n", y_test.value_counts(normalize=True))

Train shape: (4135,) Test shape: (1034,)

Train label distribution:
 label
0    0.873761
1    0.126239
Name: proportion, dtype: float64

Test label distribution:
 label
0    0.873308
1    0.126692
Name: proportion, dtype: float64


In [22]:
# ============================================================
# TEXT PREPROCESSING
# ============================================================

In [23]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase everything (so "Free" and "free" are treated the same)
    text = text.lower()
    
    # Remove non-alphabetic characters (digits, punctuation, symbols)
    # Note: this also strips things like '£' or numbers from spam messages,
    # which we already captured separately via message_length/word_count features
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Tokenize by whitespace
    tokens = text.split()
    
    # Remove stopwords (common words like "the", "is", "and" that carry
    # little discriminative signal for spam/ham classification)
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatize (reduce words to their base form, e.g. "winning" -> "win")
    # Preferred over stemming here since lemmatization produces real words,
    # which keeps the vocabulary more interpretable for feature importance later
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

# Apply cleaning to train and test separately (no leakage risk here since
# this is a pure function, not fit on data — but keeping the split intact)
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

# Compare before/after on a few examples
for original, cleaned in zip(X_train.head(3), X_train_clean.head(3)):
    print("ORIGINAL:", original)
    print("CLEANED :", cleaned)
    print("---")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...


ORIGINAL: Ta-Daaaaa! I am home babe, are you still up ?
CLEANED : ta daaaaa home babe still
---
ORIGINAL: Yup having my lunch buffet now.. U eat already?
CLEANED : yup lunch buffet u eat already
---
ORIGINAL: Ok anyway no need to change with what you said
CLEANED : ok anyway need change said
---


In [24]:
# ============================================================
# TF-IDF VECTORIZATION
# ============================================================

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF converts text into numeric vectors: each word gets a weight based on
# how frequent it is in a message (TF) vs how common it is across all messages (IDF).
# Words that appear in many messages (e.g. "call", "day") get down-weighted,
# while rare but discriminative words (e.g. "prize", "urgent") get boosted.

# max_features limits vocabulary size to the most frequent terms (controls dimensionality)
# min_df=2 ignores words that appear in only 1 message (likely noise/typos)
vectorizer = TfidfVectorizer(max_features=3000, min_df=2)

# Fit ONLY on train data, then transform both train and test
# (same principle as StandardScaler — vectorizer must not see test vocabulary)
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_test_tfidf = vectorizer.transform(X_test_clean)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

# Peek at vocabulary
print("\nSample vocabulary:", list(vectorizer.vocabulary_.keys())[:20])
print("\nVocabulary size:", len(vectorizer.vocabulary_))

Train TF-IDF shape: (4135, 2789)
Test TF-IDF shape: (1034, 2789)

Sample vocabulary: ['ta', 'home', 'babe', 'still', 'yup', 'lunch', 'buffet', 'eat', 'already', 'ok', 'anyway', 'need', 'change', 'said', 'best', 'ur', 'exam', 'later', 'reached', 'tired']

Vocabulary size: 2789


In [26]:
# ============================================================
# SMOTE (CLASS IMBALANCE HANDLING)
# ============================================================

In [33]:
from imblearn.over_sampling import SMOTE

# Applied AFTER TF-IDF vectorization (SMOTE needs numeric feature space)
# and only on the TRAIN set — test set must reflect real-world class distribution
print("Before SMOTE:", y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_tfidf, y_train)

print("After SMOTE:", y_train_resampled.value_counts().to_dict())
print("Resampled train shape:", X_train_resampled.shape)

Before SMOTE: {0: 3613, 1: 522}
After SMOTE: {0: 3613, 1: 3613}
Resampled train shape: (7226, 2789)


In [34]:
# ============================================================
# MODEL - MULTINOMIAL NAIVE BAYES
# ============================================================

In [35]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, precision_recall_curve
)

# MultinomialNB is the standard choice for text classification with TF-IDF/count features,
# since it models feature values as (weighted) word frequencies rather than
# assuming a Gaussian distribution (which GaussianNB would, incorrectly, for word counts)
nb_model = MultinomialNB()
nb_model.fit(X_train_resampled, y_train_resampled)

# Predict on the ORIGINAL (imbalanced) test set — this reflects real-world performance
y_pred = nb_model.predict(X_test_tfidf)
y_proba = nb_model.predict_proba(X_test_tfidf)[:, 1]

# Baseline metrics before threshold tuning
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("F1 (spam class):", f1_score(y_test, y_pred))

Confusion Matrix:
 [[863  40]
 [  8 123]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.96      0.97       903
           1       0.75      0.94      0.84       131

    accuracy                           0.95      1034
   macro avg       0.87      0.95      0.90      1034
weighted avg       0.96      0.95      0.96      1034

ROC-AUC: 0.9892047712037061
F1 (spam class): 0.8367346938775511


In [36]:
# ============================================================
# THRESHOLD TUNING
# ============================================================

In [37]:
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

# Default threshold is 0.5 — but for spam detection, false positives
# (real messages marked as spam) are usually more costly than false negatives
# (a spam message slipping through), so we may want to push precision up.

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Compute F1 score at each threshold to find the best balance
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold (max F1):", best_threshold)
print("Precision at best threshold:", precisions[best_idx])
print("Recall at best threshold:", recalls[best_idx])
print("F1 at best threshold:", f1_scores[best_idx])

# Apply the tuned threshold
y_pred_tuned = (y_proba >= best_threshold).astype(int)

print("\nConfusion Matrix (tuned threshold):\n", confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report (tuned threshold):\n", classification_report(y_test, y_pred_tuned))

Best threshold (max F1): 0.759768328083015
Precision at best threshold: 0.9090909090909091
Recall at best threshold: 0.916030534351145
F1 at best threshold: 0.9125475284671111

Confusion Matrix (tuned threshold):
 [[891  12]
 [ 11 120]]

Classification Report (tuned threshold):
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       903
           1       0.91      0.92      0.91       131

    accuracy                           0.98      1034
   macro avg       0.95      0.95      0.95      1034
weighted avg       0.98      0.98      0.98      1034



In [38]:
# ============================================================
# OVERFIT CHECK
# ============================================================

In [39]:
# Compare train performance (on resampled data used for training)
# vs test performance (on original imbalanced data, tuned threshold)

y_train_proba = nb_model.predict_proba(X_train_resampled)[:, 1]
y_train_pred_tuned = (y_train_proba >= best_threshold).astype(int)

train_f1 = f1_score(y_train_resampled, y_train_pred_tuned)
train_auc = roc_auc_score(y_train_resampled, y_train_proba)

test_f1 = f1_score(y_test, y_pred_tuned)
test_auc = roc_auc_score(y_test, y_proba)

print(f"Train F1: {train_f1:.4f} | Test F1: {test_f1:.4f} | Gap: {train_f1 - test_f1:.4f}")
print(f"Train AUC: {train_auc:.4f} | Test AUC: {test_auc:.4f} | Gap: {train_auc - test_auc:.4f}")

Train F1: 0.9832 | Test F1: 0.9125 | Gap: 0.0707
Train AUC: 0.9988 | Test AUC: 0.9892 | Gap: 0.0096


In [40]:
# ============================================================
# FEATURE IMPORTANCE (NB-specific: log-probability difference)
# ============================================================

In [41]:
# nb_model.feature_log_prob_ gives log P(word | class) for each class
# Shape: (n_classes, n_features) -> row 0 = ham, row 1 = spam
feature_names = vectorizer.get_feature_names_out()
log_prob_ham = nb_model.feature_log_prob_[0]
log_prob_spam = nb_model.feature_log_prob_[1]

# The difference tells us which words are most "spam-indicative" vs "ham-indicative"
# A large positive value means the word is much more likely under spam than ham
log_prob_diff = log_prob_spam - log_prob_ham

importance_df = pd.DataFrame({
    'word': feature_names,
    'log_prob_diff': log_prob_diff
}).sort_values('log_prob_diff', ascending=False)

print("Top 15 SPAM-indicative words:")
print(importance_df.head(15))

print("\nTop 15 HAM-indicative words:")
print(importance_df.tail(15))

Top 15 SPAM-indicative words:
            word  log_prob_diff
1841       prize       4.635740
438        claim       4.525570
2467        tone       4.359656
2741         www       4.244882
2529          uk       3.931105
2098     service       3.906725
1013  guaranteed       3.865579
1818         ppm       3.789498
1994    ringtone       3.626095
165      awarded       3.557988
2589       video       3.511929
2561      urgent       3.474458
1623       nokia       3.470166
146      attempt       3.462376
749        entry       3.405379

Top 15 HAM-indicative words:
           word  log_prob_diff
2029       said      -3.220865
2212  something      -3.256221
2340       sure      -3.258111
2634        wat      -3.265548
1918     really      -3.317046
1099       home      -3.318329
565          da      -3.337106
135         ask      -3.395382
95     anything      -3.480357
1366        lol      -3.504651
2045        say      -3.527906
1295      later      -3.644221
1375        lor      -4.0

In [42]:
# ============================================================
# CROSS-VALIDATION
# ============================================================

In [43]:
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Using imblearn's Pipeline (not sklearn's) because it supports SMOTE as a step
# This ensures TF-IDF and SMOTE are refit independently on each CV fold's
# training portion, preventing leakage from the validation fold
cv_pipeline = ImbPipeline([
    ('tfidf', TfidfVectorizer(max_features=3000, min_df=2)),
    ('smote', SMOTE(random_state=42)),
    ('nb', MultinomialNB())
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use the ORIGINAL uncleaned-but-split train data (raw text -> pipeline handles cleaning internally... 
# actually we pass the already-cleaned text, since clean_text() is a pure function, not fit on data)
cv_auc_scores = cross_val_score(cv_pipeline, X_train_clean, y_train, cv=skf, scoring='roc_auc')
cv_f1_scores = cross_val_score(cv_pipeline, X_train_clean, y_train, cv=skf, scoring='f1')

print(f"CV ROC-AUC: {cv_auc_scores.mean():.4f} ± {cv_auc_scores.std():.4f}")
print(f"CV F1: {cv_f1_scores.mean():.4f} ± {cv_f1_scores.std():.4f}")
print("\nIndividual fold AUC scores:", cv_auc_scores)

CV ROC-AUC: 0.9842 ± 0.0035
CV F1: 0.8473 ± 0.0136

Individual fold AUC scores: [0.9887222  0.98054979 0.97987818 0.98703337 0.98481731]
